# Dia 11 na GPU: previsões do BERT, ablação de dados e LLM

Três partes **independentes**; rode só as que precisar. Toda a lógica está em `src/` (testada); este notebook só chama os scripts.

**Antes de rodar:**
1. Acelerador **GPU T4** ligado (Ambiente de execução > Alterar o tipo de ambiente de execução).
2. Segredo **`HF_TOKEN`** (ícone de chave, com *Acesso ao notebook* ativado). Necessário para baixar o modelo privado da Parte 1.
3. O código novo precisa estar **no GitHub** (`git push`), porque o notebook faz `git clone`.

| Parte | O que faz | Tempo estimado (T4) |
| --- | --- | --- |
| 1 | previsões por exemplo do BERTimbau já treinado | poucos minutos |
| 2 | BERTimbau com 10%, 25% e 50% do treino | cerca de 25 min (estimativa pelo treino completo de 24 min) |
| 3 | LLM zero-shot e few-shot (validação, depois teste) | não medido: veja o tempo da rodada de validação antes de seguir |

No fim, baixe o `dia11_resultados.zip` e extraia na pasta `sentimento-pt/` do seu computador.

In [ ]:
import torch

assert torch.cuda.is_available(), "Sem GPU: ligue o acelerador nas configurações do notebook."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/arthurvelloso/sentimento-pt.git"
HUB_REPO = "arthurvelloso/bertimbau-sentimento-b2w"  # modelo privado do dia 10

if not os.path.exists("sentimento-pt"):
    clone = subprocess.run(
        ["git", "clone", REPO_URL],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if clone.returncode != 0:
        raise RuntimeError("Não foi possível clonar. O repositório está público?")
os.chdir("sentimento-pt")
subprocess.run(["git", "pull", "--ff-only"])
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists("src/llm_classify.py"), (
    "O código do dia 11 ainda não está no GitHub (git push)."
)

In [ ]:
# Só o que falta; o torch, o pandas e o scikit-learn já vêm instalados. Não reinstale o torch.
!pip install -q -r requirements-train.txt

In [ ]:
# Lê o token dos segredos da plataforma, sem imprimir nada.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token carregado:", bool(os.environ.get("HF_TOKEN")))

## Parte 1: previsões do BERTimbau já treinado

Baixa o modelo do Hub, prevê validação e teste e grava `results/preds_bertimbau.csv` (só ids e rótulos, sem texto). **Confira a linha de conferência**: o macro-F1 do teste recalculado deve ser o do treino (0,7608). Se não bater, o modelo ou o `max_length` não são os do treino.

In [ ]:
!python -m src.predict_bert --model $HUB_REPO

## Parte 2: ablação do tamanho dos dados do BERTimbau

Treina com 10%, 25% e 50% do treino (os mesmos tamanhos do TF-IDF); o ponto de 100% vem das previsões da Parte 1. Mesmos hiperparâmetros do treino completo. Não envia nada ao Hub.

In [ ]:
SIZES = [10094, 25236, 50472]  # 10%, 25% e 50% de 100.944 exemplos de treino

for n in SIZES:
    print(f"\n===== BERTimbau com {n} exemplos de treino =====")
    ip = get_ipython()
    ip.system(
        f"python -m src.train_bert --train-subset {n} "
        f"--results-path results/ablation_bert/n{n}.json --output-dir /tmp/ablation_{n}"
    )

In [ ]:
!python -m src.ablation_bert

## Parte 3: LLM (Qwen3-4B) em zero-shot e few-shot

**3a. Validação primeiro** (600 avaliações): serve para conferir o prompt. Olhe a **taxa de respostas inválidas** e o F1 por classe. Se mais de uns 5% das respostas forem inválidas, veja `invalid_examples` em `results/dev/*.json`, ajuste o prompt em `src/llm_classify.py` (e a `PROMPT_VERSION`) **olhando só a validação**, e rode de novo. O teste só entra na 3b, com o prompt congelado.

Anote o tempo de cada rodada (`eval_seconds` no JSON): ele define se o teste com 3.000 avaliações cabe na sua sessão.

In [ ]:
!python -m src.llm_classify --split val --shots-per-class 0
!python -m src.llm_classify --split val --shots-per-class 3

**3b. Teste** (subconjunto estratificado de 3.000 avaliações, igual para todas as rodadas). Rode **uma vez**, com o prompt congelado. Para o teste inteiro, acrescente `--max-eval 12618` (bem mais lento).

In [ ]:
!python -m src.llm_classify --split test --shots-per-class 0
!python -m src.llm_classify --split test --shots-per-class 3

## Levar os resultados de volta

Junta tudo num `.zip`. Extraia-o dentro da pasta `sentimento-pt/` do seu computador (os caminhos já são relativos a ela).

In [ ]:
import glob
import zipfile

wanted = [
    "results/preds_bertimbau.csv",
    "results/ablation_bertimbau.json",
    *glob.glob("results/ablation_bert/*.json"),
    *glob.glob("results/llm_*.json"),
    *glob.glob("results/preds_llm_*.csv"),
    *glob.glob("results/dev/*"),
]
present = [path for path in wanted if os.path.exists(path)]
with zipfile.ZipFile("dia11_resultados.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in present:
        archive.write(path)
print("no zip:", *present, sep="\n  ")

try:
    from google.colab import files as colab_files

    colab_files.download("dia11_resultados.zip")
except ImportError:
    print("Fora do Colab: baixe o dia11_resultados.zip pelo painel de arquivos.")